# Meta-Learner Feature Engineering And XGBoost Classifier


## Configuration


## Imports


In [41]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier

from meta_learner_classifier_tsplit_utils import (
    TARGET_CLASS_COL,
    TemporalDecaySampleWeightClassifier,
    build_classifier_scoring,
    build_meta_classifier_dataset_from_feature_frame,
    evaluate_baseline_hard_class_over_splits,
    fit_best_xgb_classifier,
    print_classifier_cv_metrics,
    select_best_trial_lexicographic_classifier,
    summarize_classifier_lexicographic_candidates,
    summarize_classifier_optuna_trials,
    summarize_classifier_predictions,
    summarize_confidence_thresholds,
    summarize_day_by_day_classifier_thresholds,
    summarize_hard_class_predictions,
    tune_xgb_classifier_optuna,
)
from meta_learner_feature_utils import (
    build_meta_learner_feature_frame,
    load_meta_learner_dataframe,
)
from nba_ou.modeling.modeling import (
    ModelBundleMetadata,
    ModelInfo,
    TrainingMetrics,
    assert_valid_time_splits,
    evaluate_day_by_day_walk_forward,
    make_test_anchored_walk_forward_splits,
    save_model_bundle,
    split_latest_dates_holdout,
)

pd.options.display.float_format = '{:,.4f}'.format


In [42]:
CSV_PATH = '/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/meta_learner_last_2_seasons_20260405.csv'
OUTPUT_FEATURE_CSV = None

ROLLING_WINDOW_DAYS = 5
LINE_BUCKET_EDGES = [205.0, 215.0, 225.0, 235.0, 245.0]
CONFIDENCE_BUCKET_EDGES = [1.0, 2.0, 3.0, 5.0]
INCLUDE_BASELINE_FEATURES = False
FINAL_HOLDOUT_SIZE = 0.08

TRAIN_GAMES = 1000
TEST_GAMES = 20
STEP_GAMES_BETWEEN_TESTS = 40
MIN_TRAIN_GAMES = int(TRAIN_GAMES * 0.75)
MAX_FOLDS = 15

SAMPLE_WEIGHT_LAMBDA = 0.0075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.02)
CONFIDENCE_THRESHOLDS = (0.50, 0.52, 0.55, 0.58, 0.60, 0.65)

XGB_FIXED_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'max_depth': 3,
    'learning_rate': 0.05,
    'n_estimators': 60,
    'subsample': 0.75,
    'colsample_bytree': 0.80,
    'reg_alpha': 2.0,
    'reg_lambda': 2.0,
    'min_child_weight': 4,
    'gamma': 0.0,
    'n_jobs': -1,
    'random_state': 16,
}

OPTUNA_N_TRIALS = 80
OPTUNA_TIMEOUT_SECONDS = int(4.5 * 3600)
OPTUNA_LOG_LOSS_TOLERANCE_ABS = 0.01
MODEL_OUT_DIR = '/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/meta_learner/classifier/'


## Load Data


In [43]:
raw_df = load_meta_learner_dataframe(CSV_PATH)

feature_result = build_meta_learner_feature_frame(
    raw_df,
    rolling_window_days=ROLLING_WINDOW_DAYS,
    line_bucket_edges=LINE_BUCKET_EDGES,
    confidence_bucket_edges=CONFIDENCE_BUCKET_EDGES,
)

df_features = feature_result.dataframe.copy()

print(f'CSV path: {CSV_PATH}')
print(f'Rows in feature frame: {len(df_features):,}')
print(f'Columns in feature frame: {len(df_features.columns):,}')
print(f'Line column: {feature_result.line_col}')
print(f'Total-points model columns: {feature_result.total_model_cols}')
print(f'Line-error model columns: {feature_result.line_error_model_cols}')
print(f'Error-space prediction columns: {feature_result.error_cols}')
print(f'Vote feature count: {len(feature_result.vote_feature_cols)}')
print(f'Reliability feature count: {len(feature_result.reliability_feature_cols)}')


CSV path: /home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/meta_learner_last_2_seasons_20260405.csv
Rows in feature frame: 2,317
Columns in feature frame: 364
Line column: TOTAL_LINE_bet365
Total-points model columns: ['PRED_TOTAL_POINTS_FULL_DATASET', 'PRED_TOTAL_POINTS_LAST_5_SEASONS', 'PRED_TOTAL_POINTS_LAST_3_SEASONS']
Line-error model columns: ['PRED_LINE_ERROR_FULL_DATASET', 'PRED_LINE_ERROR_LAST_5_SEASONS', 'PRED_LINE_ERROR_LAST_3_SEASONS']
Error-space prediction columns: ['PRED_TOTAL_POINTS_FULL_DATASET__ERR', 'PRED_TOTAL_POINTS_LAST_5_SEASONS__ERR', 'PRED_TOTAL_POINTS_LAST_3_SEASONS__ERR', 'PRED_LINE_ERROR_FULL_DATASET__ERR', 'PRED_LINE_ERROR_LAST_5_SEASONS__ERR', 'PRED_LINE_ERROR_LAST_3_SEASONS__ERR']
Vote feature count: 14
Reliability feature count: 67


In [44]:
display(pd.DataFrame({'vote_feature': feature_result.vote_feature_cols}))

example_model = feature_result.error_cols[0]
example_reliability_cols = [
    column
    for column in feature_result.reliability_feature_cols
    if column == 'META_TOTAL_LINE_BUCKET' or column.startswith(example_model)
]
display(pd.DataFrame({'example_reliability_feature': example_reliability_cols}))

preview_cols = [
    'GAME_ID',
    'GAME_DATE',
    'SEASON_YEAR',
    'TOTAL_POINTS',
    'LINE_ERROR',
    feature_result.line_col,
    *feature_result.error_cols,
    *feature_result.vote_feature_cols,
    *example_reliability_cols[:8],
    *feature_result.baseline_cols,
]
preview_cols = [column for column in preview_cols if column in df_features.columns]
display(df_features[preview_cols].head(10))


,vote_feature
0,META_VALID_VOTE_COUNT
1,META_OVER_VOTE_COUNT
2,META_UNDER_VOTE_COUNT
3,META_PUSH_VOTE_COUNT
4,META_VOTE_PROP_OVER
5,META_VOTE_PROP_UNDER
6,META_VOTE_PROP_PUSH
7,META_VOTE_MEAN_ERR
8,META_VOTE_MEDIAN_ERR
9,META_VOTE_STD_ERR


,example_reliability_feature
0,META_TOTAL_LINE_BUCKET
1,PRED_TOTAL_POINTS_FULL_DATASET__ERR_CONF_BUCKET
2,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_HI...
3,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_MA...
4,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_BE...
5,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_BI...
6,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_LI...
7,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_LI...
8,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_LI...
9,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_CO...


,GAME_ID,GAME_DATE,SEASON_YEAR,TOTAL_POINTS,LINE_ERROR,TOTAL_LINE_bet365,PRED_TOTAL_POINTS_FULL_DATASET__ERR,PRED_TOTAL_POINTS_LAST_5_SEASONS__ERR,PRED_TOTAL_POINTS_LAST_3_SEASONS__ERR,PRED_LINE_ERROR_FULL_DATASET__ERR,...,META_TOTAL_LINE_BUCKET,PRED_TOTAL_POINTS_FULL_DATASET__ERR_CONF_BUCKET,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_HISTORY_N_5D,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_MAE_5D,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_BETTING_ACC_5D,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_BIAS_5D,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_LINE_BUCKET_N_5D,PRED_TOTAL_POINTS_FULL_DATASET__ERR_ROLLING_LINE_BUCKET_MAE_5D,BASE_AVG_ALL_6_ERR,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_FULL_DATASET_ERR
0,0022400124,2024-10-30,2024,225,-1.0000,226.0000,5.0608,1.9623,4.2785,-0.1026,...,3.0000,4.0000,0.0000,NaN,NaN,NaN,0.0000,NaN,1.9777,1.0000
1,0022400126,2024-10-30,2024,193,-26.0000,219.0000,0.1665,0.6270,1.1051,-2.1152,...,2.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,NaN,-0.3129,1.0000
2,0022400128,2024-10-31,2024,221,-9.5000,230.5000,-2.2423,-0.2777,1.1707,0.1334,...,3.0000,2.0000,2.0000,16.1136,0.0000,16.1136,1.0000,6.0608,0.5495,1.0000
3,0022400129,2024-10-31,2024,210,-16.0000,226.0000,0.9377,-0.1740,-1.5470,-2.2162,...,3.0000,0.0000,2.0000,16.1136,0.0000,16.1136,1.0000,6.0608,-1.0159,-1.0000
4,0022400130,2024-10-31,2024,194,-25.5000,219.5000,-2.8246,2.1607,0.7031,-2.3331,...,2.0000,2.0000,2.0000,16.1136,0.0000,16.1136,1.0000,26.1665,-0.5528,-1.0000
5,0022400131,2024-10-31,2024,244,24.0000,220.0000,-7.3328,-7.0838,-5.1081,-2.6081,...,2.0000,4.0000,2.0000,16.1136,0.0000,16.1136,1.0000,26.1665,-5.2165,-1.0000
6,0022400132,2024-11-01,2024,233,0.0000,233.0000,-3.2967,-1.9133,-1.6573,-3.0974,...,3.0000,3.0000,6.0000,18.4051,0.3333,7.9609,3.0000,10.0854,-2.1547,-1.0000
7,0022400133,2024-11-01,2024,229,13.0000,216.0000,0.2521,1.4298,4.4885,-4.1308,...,2.0000,0.0000,6.0000,18.4051,0.3333,7.9609,3.0000,26.7249,0.5759,1.0000
8,0022400134,2024-11-01,2024,226,6.5000,219.5000,-5.7079,-4.5014,-2.1868,-0.8739,...,2.0000,4.0000,6.0000,18.4051,0.3333,7.9609,3.0000,26.7249,-2.8090,-1.0000
9,0022400135,2024-11-01,2024,238,2.0000,236.0000,-4.2478,-5.2673,-1.1169,-2.1675,...,4.0000,3.0000,6.0000,18.4051,0.3333,7.9609,0.0000,NaN,-2.0208,-1.0000


In [45]:
if OUTPUT_FEATURE_CSV is not None:
    output_path = Path(OUTPUT_FEATURE_CSV)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_features.to_csv(output_path, index=False)
    print(f'Saved engineered feature frame to: {output_path}')
else:
    print('Set OUTPUT_FEATURE_CSV to save the engineered dataframe.')


Set OUTPUT_FEATURE_CSV to save the engineered dataframe.


## Train / Test


In [46]:
dataset = build_meta_classifier_dataset_from_feature_frame(
    df_features,
    include_baseline_features=INCLUDE_BASELINE_FEATURES,
)

df_model = dataset.dataframe.reset_index(drop=True).copy()
X_all = dataset.feature_frame.reset_index(drop=True).copy()
y_all = dataset.target.reset_index(drop=True).copy()

print(f'Rows after removing pushes: {len(df_model):,}')
print('Season counts:')
print(df_model['SEASON_YEAR'].value_counts().sort_index())
print(f'Feature count kept: {len(dataset.feature_cols)}')
print(f'Baseline features included as model inputs: {INCLUDE_BASELINE_FEATURES}')
print(f'Dropped non-numeric feature columns: {len(dataset.dropped_feature_cols)}')
if dataset.dropped_feature_cols:
    print(dataset.dropped_feature_cols[:20])


Rows after removing pushes: 2,293
Season counts:
SEASON_YEAR
2024    1219
2025    1074
Name: count, dtype: int64
Feature count kept: 355
Baseline features included as model inputs: False
Dropped non-numeric feature columns: 0


In [47]:
df_split = df_model.copy()
df_split['__row_id__'] = np.arange(len(df_split))

df_dev, df_test_final = split_latest_dates_holdout(
    df=df_split,
    date_col='GAME_DATE',
    test_size=FINAL_HOLDOUT_SIZE,
)

dev_idx = df_dev['__row_id__'].to_numpy()
test_idx = df_test_final['__row_id__'].to_numpy()

X_dev = X_all.iloc[dev_idx].copy()
X_test_final = X_all.iloc[test_idx].copy()
y_dev = y_all.iloc[dev_idx].copy()
y_test_final = y_all.iloc[test_idx].copy()

print(f'Development rows: {len(df_dev):,}')
print(f'Final holdout rows: {len(df_test_final):,}')
print('Final holdout date range:', df_test_final['GAME_DATE'].min(), '->', df_test_final['GAME_DATE'].max())
print('Development season counts:')
print(df_dev['SEASON_YEAR'].value_counts().sort_index())
print('Final holdout season counts:')
print(df_test_final['SEASON_YEAR'].value_counts().sort_index())


Development rows: 2,106
Final holdout rows: 187
Final holdout date range: 2026-03-12 00:00:00 -> 2026-04-05 00:00:00
Development season counts:
SEASON_YEAR
2024    1219
2025     887
Name: count, dtype: int64
Final holdout season counts:
SEASON_YEAR
2025    187
Name: count, dtype: int64


In [48]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col='GAME_DATE',
    season_col='SEASON_YEAR',
    test_games=TEST_GAMES,
    step_games_between_tests=STEP_GAMES_BETWEEN_TESTS,
    train_games=TRAIN_GAMES,
    min_train_games=MIN_TRAIN_GAMES,
    max_folds=MAX_FOLDS,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits, date_col='GAME_DATE')
display(fold_info)


Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           1000            21       2024-11-12     2025-04-06      2025-04-07    2025-04-09         2024
    2           1000            23       2024-11-20     2025-04-22      2025-04-23    2025-04-29         2024
    3           1000            24       2024-11-30     2025-06-22      2025-10-27    2025-11-02         2025
    4           1000            22       2024-12-11     2025-11-08      2025-11-09    2025-11-11         2025
    5           1000            28       2024-12-23     2025-11-17      2025-11-18    2025-11-21         2025
    6           1000            25       2025-01-04     2025-11-28      2025-11-29    2025-12-01         2025
    7           1000            20       2025-01-14     2025-12-07      2025-12-08    2025-12-13         2025
    8           1000            27       2025-01-22     2025-12-20      2025

,fold,train_n_games,test_n_games,train_start_date,train_end_date,test_start_date,test_end_date,test_season
0,1,1000,21,2024-11-12,2025-04-06,2025-04-07,2025-04-09,2024
1,2,1000,23,2024-11-20,2025-04-22,2025-04-23,2025-04-29,2024
2,3,1000,24,2024-11-30,2025-06-22,2025-10-27,2025-11-02,2025
3,4,1000,22,2024-12-11,2025-11-08,2025-11-09,2025-11-11,2025
4,5,1000,28,2024-12-23,2025-11-17,2025-11-18,2025-11-21,2025
5,6,1000,25,2025-01-04,2025-11-28,2025-11-29,2025-12-01,2025
6,7,1000,20,2025-01-14,2025-12-07,2025-12-08,2025-12-13,2025
7,8,1000,27,2025-01-22,2025-12-20,2025-12-21,2025-12-23,2025
8,9,1000,28,2025-02-01,2025-12-29,2025-12-30,2026-01-02,2025
9,10,1000,29,2025-02-10,2026-01-07,2026-01-08,2026-01-11,2025


## Baselines


In [49]:
baseline_cv_summary = evaluate_baseline_hard_class_over_splits(
    df=df_dev,
    target_col=TARGET_CLASS_COL,
    baseline_error_cols=dataset.baseline_cols,
    splits=splits,
)

baseline_final_summary = pd.DataFrame(
    [
        summarize_hard_class_predictions(
            y_true=y_test_final,
            y_pred=(pd.to_numeric(df_test_final[column], errors='coerce') > 0).astype(int),
            label=column,
        )
        for column in dataset.baseline_cols
    ]
)

print('Cross-validation baselines')
display(baseline_cv_summary.round(4))
print('Final holdout baselines')
display(baseline_final_summary.round(4))


Cross-validation baselines


,model,validation_accuracy_pct,validation_balanced_accuracy_pct
0,BASE_AVG_ALL_6_ERR,54.9957,51.5432
1,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9389,52.2239


Final holdout baselines


,model,n_games,accuracy_pct,balanced_accuracy_pct
0,BASE_AVG_ALL_6_ERR,187,50.2674,50.8484
1,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,187,52.4064,52.7861


In [50]:
SCORING = build_classifier_scoring()
DAY_BY_DAY_METRIC_NAME = 'Accuracy'


def print_final_metrics(label, y_true, y_proba):
    summary = pd.DataFrame(
        [
            summarize_classifier_predictions(
                y_true=y_true,
                y_proba=y_proba,
                label=label,
            )
        ]
    )
    display(summary.round(4))
    return summary


def run_day_by_day_classifier_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    thresholds=CONFIDENCE_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: accuracy_score(
            np.asarray(y_true, dtype=int),
            (np.asarray(y_pred, dtype=float) >= 0.5).astype(int),
        ),
        target_col=TARGET_CLASS_COL,
        date_col='GAME_DATE',
        max_games=max_games,
        metric_name=DAY_BY_DAY_METRIC_NAME,
    )

    threshold_results = summarize_day_by_day_classifier_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f'{label} mean day-by-day {DAY_BY_DAY_METRIC_NAME}: {result.mean_metric:.2%}')
    display(result.daily_results.style.format({DAY_BY_DAY_METRIC_NAME: '{:.2%}'}))
    print(f'{label} thresholded walk-forward accuracy')
    display(
        threshold_results.style.format(
            {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
        )
    )
    return result, threshold_results

## XGBoost No Sample Weights


In [51]:
xgb_clf_no_weights = XGBClassifier(**XGB_FIXED_PARAMS)

cv_results_no_weights = cross_validate(
    xgb_clf_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1,
)

print('XGBoost classifier no sample weights')
print_classifier_cv_metrics(cv_results_no_weights, SCORING)

XGBoost classifier no sample weights
Train Accuracy: 89.41%
Validation Accuracy: 52.50%

Train Balanced_Accuracy: 89.30%
Validation Balanced_Accuracy: 53.06%

Train Brier: 0.17697
Validation Brier: 0.24953

Train LogLoss: 0.54349
Validation LogLoss: 0.69243



In [52]:
xgb_clf_no_weights.fit(X_dev, y_dev)
y_proba_test_no_weights = xgb_clf_no_weights.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_no_weights = print_final_metrics(
    'XGBoost classifier no sample weights',
    y_test_final,
    y_proba_test_no_weights,
)

Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,XGBoost classifier no sample weights,187,54.0107,54.5230,0.2485,0.6901,48.6990,54.8662


In [53]:
threshold_summary_no_weights = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_no_weights,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_no_weights.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)

,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,187,100.0%,54.01,54.87
1,0.520000,131,70.1%,54.20,56.59
2,0.550000,88,47.1%,53.41,58.22
3,0.580000,40,21.4%,62.50,60.48
4,0.600000,18,9.6%,61.11,62.20
5,0.650000,3,1.6%,66.67,65.88


In [54]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBClassifier(**XGB_FIXED_PARAMS)

    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_classifier_evaluation(
    label='XGBoost classifier no sample weights',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost classifier no sample weights mean day-by-day Accuracy: 54.42%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-12 00:00:00,1000,8,2025-04-09 00:00:00,2026-03-11 00:00:00,62.50%
1,2026-03-13 00:00:00,1000,8,2025-04-11 00:00:00,2026-03-12 00:00:00,62.50%
2,2026-03-14 00:00:00,1000,7,2025-04-11 00:00:00,2026-03-13 00:00:00,71.43%
3,2026-03-15 00:00:00,1000,7,2025-04-13 00:00:00,2026-03-14 00:00:00,42.86%
4,2026-03-16 00:00:00,1000,7,2025-04-13 00:00:00,2026-03-15 00:00:00,42.86%
5,2026-03-17 00:00:00,1000,8,2025-04-18 00:00:00,2026-03-16 00:00:00,62.50%
6,2026-03-18 00:00:00,1000,9,2025-04-23 00:00:00,2026-03-17 00:00:00,11.11%
7,2026-03-19 00:00:00,1000,8,2025-04-26 00:00:00,2026-03-18 00:00:00,37.50%
8,2026-03-20 00:00:00,1000,6,2025-04-28 00:00:00,2026-03-19 00:00:00,83.33%
9,2026-03-21 00:00:00,1000,10,2025-04-30 00:00:00,2026-03-20 00:00:00,40.00%


XGBoost classifier no sample weights thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,187,100.0%,52.94,57.42
1,0.520000,147,78.6%,53.06,59.19
2,0.550000,108,57.8%,50.93,61.21
3,0.580000,81,43.3%,50.62,62.76
4,0.600000,60,32.1%,50.00,64.17
5,0.650000,22,11.8%,59.09,67.54


## Check Weighted


In [55]:
xgb_clf_weighted = XGBClassifier(**XGB_FIXED_PARAMS)
weighted_xgb_clf = TemporalDecaySampleWeightClassifier(
    estimator=xgb_clf_weighted,
    dates=df_dev['GAME_DATE'],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb_clf,
    X_dev,
    y_dev,
    cv=splits,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=1,
)

print('XGBoost classifier with sample weights')
print_classifier_cv_metrics(cv_results_weights, SCORING)

XGBoost classifier with sample weights
Train Accuracy: 78.51%
Validation Accuracy: 54.33%

Train Balanced_Accuracy: 78.28%
Validation Balanced_Accuracy: 53.80%

Train Brier: 0.19697
Validation Brier: 0.24698

Train LogLoss: 0.58399
Validation LogLoss: 0.68723



In [56]:
weighted_xgb_clf.fit(X_dev, y_dev)
y_proba_test_weights = weighted_xgb_clf.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_weights = print_final_metrics(
    'XGBoost classifier with sample weights',
    y_test_final,
    y_proba_test_weights,
)

Final holdout metrics


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,XGBoost classifier with sample weights,187,54.0107,54.7294,0.2483,0.6896,46.8000,57.0762


In [57]:
threshold_summary_weights = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_weights,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_weights.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)

,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,187,100.0%,54.01,57.08
1,0.520000,152,81.3%,55.26,58.49
2,0.550000,108,57.8%,56.48,60.53
3,0.580000,75,40.1%,57.33,62.40
4,0.600000,49,26.2%,59.18,64.20
5,0.650000,19,10.2%,57.89,67.46


In [58]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBClassifier(**XGB_FIXED_PARAMS)

    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model = TemporalDecaySampleWeightClassifier(
        estimator=base_model,
        dates=train_df['GAME_DATE'],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_classifier_evaluation(
    label='XGBoost classifier with sample weights',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost classifier with sample weights mean day-by-day Accuracy: 53.49%


,date,train_n_games,test_n_games,train_start_date,train_end_date,Accuracy
0,2026-03-12 00:00:00,1000,8,2025-04-09 00:00:00,2026-03-11 00:00:00,50.00%
1,2026-03-13 00:00:00,1000,8,2025-04-11 00:00:00,2026-03-12 00:00:00,62.50%
2,2026-03-14 00:00:00,1000,7,2025-04-11 00:00:00,2026-03-13 00:00:00,85.71%
3,2026-03-15 00:00:00,1000,7,2025-04-13 00:00:00,2026-03-14 00:00:00,42.86%
4,2026-03-16 00:00:00,1000,7,2025-04-13 00:00:00,2026-03-15 00:00:00,42.86%
5,2026-03-17 00:00:00,1000,8,2025-04-18 00:00:00,2026-03-16 00:00:00,50.00%
6,2026-03-18 00:00:00,1000,9,2025-04-23 00:00:00,2026-03-17 00:00:00,11.11%
7,2026-03-19 00:00:00,1000,8,2025-04-26 00:00:00,2026-03-18 00:00:00,62.50%
8,2026-03-20 00:00:00,1000,6,2025-04-28 00:00:00,2026-03-19 00:00:00,66.67%
9,2026-03-21 00:00:00,1000,10,2025-04-30 00:00:00,2026-03-20 00:00:00,20.00%


XGBoost classifier with sample weights thresholded walk-forward accuracy


,threshold_confidence_gte,n_games,pct_of_test,bet_accuracy_pct,mean_confidence_pct
0,0.500000,187,100.0%,52.94,57.75
1,0.520000,153,81.8%,50.98,59.28
2,0.550000,108,57.8%,48.15,61.69
3,0.580000,80,42.8%,52.50,63.49
4,0.600000,64,34.2%,48.44,64.63
5,0.650000,25,13.4%,48.00,68.56


In [59]:
cv_classifier_summary = pd.DataFrame(
    [
        {
            'model': 'XGBoost classifier no sample weights',
            'validation_accuracy_pct': 100.0 * cv_results_no_weights['test_Accuracy'].mean(),
            'validation_balanced_accuracy_pct': 100.0 * cv_results_no_weights['test_Balanced_Accuracy'].mean(),
            'validation_brier_score': -cv_results_no_weights['test_Brier'].mean(),
            'validation_log_loss': -cv_results_no_weights['test_LogLoss'].mean(),
        },
        {
            'model': 'XGBoost classifier with sample weights',
            'validation_accuracy_pct': 100.0 * cv_results_weights['test_Accuracy'].mean(),
            'validation_balanced_accuracy_pct': 100.0 * cv_results_weights['test_Balanced_Accuracy'].mean(),
            'validation_brier_score': -cv_results_weights['test_Brier'].mean(),
            'validation_log_loss': -cv_results_weights['test_LogLoss'].mean(),
        },
    ]
)

final_classifier_summary = pd.concat(
    [final_summary_no_weights, final_summary_weights],
    ignore_index=True,
)

print('Cross-validation comparison')
display(
    pd.concat(
        [baseline_cv_summary, cv_classifier_summary],
        ignore_index=True,
    ).round(4)
)

print('Final holdout comparison')
display(
    pd.concat(
        [
            baseline_final_summary,
            final_classifier_summary,
        ],
        ignore_index=True,
    ).round(4)
)


Cross-validation comparison


,model,validation_accuracy_pct,validation_balanced_accuracy_pct,validation_brier_score,validation_log_loss
0,BASE_AVG_ALL_6_ERR,54.9957,51.5432,NaN,NaN
1,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,55.9389,52.2239,NaN,NaN
2,XGBoost classifier no sample weights,52.4952,53.0560,0.2495,0.6924
3,XGBoost classifier with sample weights,54.3343,53.8045,0.2470,0.6872


Final holdout comparison


,model,n_games,accuracy_pct,balanced_accuracy_pct,brier_score,log_loss,mean_predicted_over_pct,mean_confidence_pct
0,BASE_AVG_ALL_6_ERR,187,50.2674,50.8484,NaN,NaN,NaN,NaN
1,BASE_MAJORITY_ALL_6_TIE_WITH_PRED_LINE_ERROR_F...,187,52.4064,52.7861,NaN,NaN,NaN,NaN
2,XGBoost classifier no sample weights,187,54.0107,54.5230,0.2485,0.6901,48.6990,54.8662
3,XGBoost classifier with sample weights,187,54.0107,54.7294,0.2483,0.6896,46.8000,57.0762


# Optuna


In [ ]:
study = tune_xgb_classifier_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev['GAME_DATE'],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=OPTUNA_N_TRIALS,
    timeout=OPTUNA_TIMEOUT_SECONDS,
    study_name='meta_learner_xgb_classifier_logloss',
)

best_trial_lexi = select_best_trial_lexicographic_classifier(
    study,
    log_loss_tolerance_abs=OPTUNA_LOG_LOSS_TOLERANCE_ABS,
)

print('Optuna best by log loss only')
print('Trial:', study.best_trial.number)
print('Best CV log loss:', study.best_value)
print('Mean accuracy:', study.best_trial.user_attrs.get('mean_accuracy'))
print('Mean balanced accuracy:', study.best_trial.user_attrs.get('mean_balanced_accuracy'))
print('Mean Brier:', study.best_trial.user_attrs.get('mean_brier'))

print('\nSelected trial after log-loss-first / accuracy-second ranking')
print('Trial:', best_trial_lexi.number)
print('CV log loss:', best_trial_lexi.user_attrs.get('mean_log_loss', best_trial_lexi.value))
print('Mean accuracy:', best_trial_lexi.user_attrs.get('mean_accuracy'))
print('Mean balanced accuracy:', best_trial_lexi.user_attrs.get('mean_balanced_accuracy'))
print('Mean Brier:', best_trial_lexi.user_attrs.get('mean_brier'))
print('Median best_iteration:', best_trial_lexi.user_attrs.get('median_best_iteration'))
print('Params:')
for k, v in best_trial_lexi.params.items():
    print(f'{k}: {v}')

trials_df = summarize_classifier_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            'value_log_loss': '{:.4f}',
            'mean_log_loss': '{:.4f}',
            'mean_accuracy': '{:.2%}',
            'mean_balanced_accuracy': '{:.2%}',
            'mean_brier': '{:.4f}',
        }
    )
)

candidates_df = summarize_classifier_lexicographic_candidates(
    study,
    log_loss_tolerance_abs=OPTUNA_LOG_LOSS_TOLERANCE_ABS,
)
display(
    candidates_df.head(15).style.format(
        {
            'value_log_loss': '{:.4f}',
            'mean_log_loss': '{:.4f}',
            'mean_accuracy': '{:.2%}',
            'mean_balanced_accuracy': '{:.2%}',
            'mean_brier': '{:.4f}',
        }
    )
)


In [ ]:
optuna_model = fit_best_xgb_classifier(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=df_dev['GAME_DATE'],
    sample_weight_lambda=best_trial_lexi.params.get('sample_weight_lambda'),
    trial=best_trial_lexi,
)

y_proba_test_optuna = optuna_model.predict_proba(X_test_final)[:, 1]

print('Final holdout metrics')
final_summary_optuna = print_final_metrics(
    'Optuna-selected XGBoost classifier',
    y_test_final,
    y_proba_test_optuna,
)


In [ ]:
threshold_summary_optuna = summarize_confidence_thresholds(
    y_true=y_test_final,
    y_proba=y_proba_test_optuna,
    thresholds=CONFIDENCE_THRESHOLDS,
)

display(
    threshold_summary_optuna.style.format(
        {'pct_of_test': '{:.1%}', 'bet_accuracy_pct': '{:.2f}', 'mean_confidence_pct': '{:.2f}'}
    )
)


In [ ]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    train_rows = train_df['__row_id__'].to_numpy(dtype=int)
    test_rows = test_df['__row_id__'].to_numpy(dtype=int)

    X_train = X_all.iloc[train_rows]
    y_train = y_all.iloc[train_rows]
    X_test = X_all.iloc[test_rows]

    model = fit_best_xgb_classifier(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df['GAME_DATE'],
        sample_weight_lambda=best_trial_lexi.params.get('sample_weight_lambda'),
        trial=best_trial_lexi,
    )
    return model.predict_proba(X_test)[:, 1]


day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_classifier_evaluation(
    label='Optuna-selected XGBoost classifier',
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)


In [ ]:
optuna_cv_summary = pd.DataFrame(
    [
        {
            'model': 'Optuna-selected XGBoost classifier',
            'validation_accuracy_pct': 100.0 * float(best_trial_lexi.user_attrs.get('mean_accuracy', np.nan)),
            'validation_balanced_accuracy_pct': 100.0 * float(best_trial_lexi.user_attrs.get('mean_balanced_accuracy', np.nan)),
            'validation_brier_score': float(best_trial_lexi.user_attrs.get('mean_brier', np.nan)),
            'validation_log_loss': float(best_trial_lexi.user_attrs.get('mean_log_loss', np.nan)),
        }
    ]
)

print('Cross-validation comparison with Optuna')
display(
    pd.concat(
        [baseline_cv_summary, cv_classifier_summary, optuna_cv_summary],
        ignore_index=True,
    ).round(4)
)

print('Final holdout comparison with Optuna')
display(
    pd.concat(
        [baseline_final_summary, final_classifier_summary, final_summary_optuna],
        ignore_index=True,
    ).round(4)
)


In [ ]:
df_to_train_split_rows = df_model.copy().tail(TRAIN_GAMES)

X_full = X_all.loc[df_to_train_split_rows.index].copy()
y_full = y_all.loc[df_to_train_split_rows.index].copy()
sample_weight_dates_full = df_to_train_split_rows['GAME_DATE']

production_model = fit_best_xgb_classifier(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get('sample_weight_lambda'),
    trial=best_trial_lexi,
)

latest_training_date = pd.to_datetime(df_to_train_split_rows['GAME_DATE']).max()
model_version = latest_training_date.strftime('%d_%m_%y')
model_name = f'meta_learner_xgb_classifier_{model_version}'

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type='meta_learner_classifier',
        prediction_source='meta_learner_xgb_classifier',
        training_code_tag='1.0',
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get('mean_best_iteration'),
        median_best_iteration=best_trial_lexi.user_attrs.get('median_best_iteration'),
        train_games=TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
        cv_mae=float(best_trial_lexi.user_attrs.get('mean_log_loss', best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get('mean_brier'),
        cv_ou_acc=best_trial_lexi.user_attrs.get('mean_accuracy'),
        final_test_mae=float(final_summary_optuna.iloc[0]['log_loss']),
        final_test_rmse=float(final_summary_optuna.iloc[0]['brier_score']),
        final_test_ou_acc=float(final_summary_optuna.iloc[0]['accuracy_pct'] / 100.0),
        nan_threshold=0.0,
        max_na_per_row=0,
        train_date_min=df_to_train_split_rows['GAME_DATE'].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows['GAME_DATE'].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir=MODEL_OUT_DIR,
    metadata=metadata,
)

print(
    f'Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration.'
)
print('Saved model :', model_path)
print('Saved metadata:', meta_path)
